#Prep Environment

In [1]:
# Installing ollama
!apt-get install -y -qq zstd > /dev/null 2>&1 || (apt-get update -qq > /dev/null 2>&1 && apt-get install -y -qq zstd > /dev/null 2>&1)
!curl -fsSL https://ollama.com/install.sh | sh 2>&1 | tail -n 2

>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [2]:
# Running ollama server
import subprocess
import time
import requests

OLLAMA_URL = "http://localhost:11434"

def server_is_up():
    try:
      return requests.get(f"{OLLAMA_URL}/api/version", timeout=2).ok
    except requests.RequestException:
      return False

if not server_is_up():
    subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(30):
        if server_is_up(): break
        time.sleep(1)

print("Server running:", server_is_up())

Server running: True


In [3]:
# Pull models
!ollama pull qwen2.5:3b
!ollama pull nomic-embed-text
!ollama list



NAME                       ID              SIZE      MODIFIED               
nomic-embed-text:latest    0a109f422b47    274 MB    Less than a second ago    
qwen2.5:3b                 357c53fb659c    1.9 GB    8 seconds ago             


#LLM Client

In [4]:
!pip -q install chromadb openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 87.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135

In [5]:
import os, re, glob, json
from openai import OpenAI
import chromadb

OLLAMA_URL = "http://localhost:11434"
GEN_MODEL  = "qwen2.5:3b"
EMB_MODEL  = "nomic-embed-text"
CORPUS_DIR = "corpus/md"
CHROMA_DIR = "chroma"

client = OpenAI(base_url=f"{OLLAMA_URL}/v1", api_key="ollama")

#Embbedings

In [6]:
def embed(texts, model=EMB_MODEL):
    r = requests.post(f"{OLLAMA_URL}/api/embed", json={"model": model, "input": texts})
    r.raise_for_status()
    return r.json()["embeddings"]

def build_index(rows, name, persist_dir=CHROMA_DIR, batch=32, rebuild=False):
    store = chromadb.PersistentClient(path=persist_dir)
    if rebuild:
        try: store.delete_collection(name)
        except Exception: pass
    col = store.get_or_create_collection(name, metadata={"hnsw:space": "cosine", "embedding_model": EMB_MODEL})
    if col.count() == len(rows):
        return col
    for i in range(0, len(rows), batch):
        part = rows[i:i + batch]
        col.upsert(ids=[r["id"] for r in part], documents=[r["text"] for r in part],
                   embeddings=embed([r["text"] for r in part]),
                   metadatas=[{"doc": r["doc"], "chunk": r["chunk"], "section": r["section"]} for r in part])
    return col

def retrieve(col, question, k=4):
    assert col.metadata.get("embedding_model") == EMB_MODEL, "índice e consulta com modelos de embedding diferentes"
    res = col.query(query_embeddings=embed([question]), n_results=k, include=["documents", "metadatas", "distances"])
    return [{"text": t, "doc": m["doc"], "section": m["section"], "distance": d}
            for t, m, d in zip(res["documents"][0], res["metadatas"][0], res["distances"][0])]

#Corpus

Example corpus

In [7]:
os.makedirs(CORPUS_DIR, exist_ok=True)

CORPUS = {
    "datas_e_prazos.md": (
        "# Eleições Gerais 2026 · Guia de Calendário · Seção 1 · Datas e Prazos"
        " Oficiais\n\nEste documento detalha o cronograma oficial para as"
        " Eleições Gerais de 2026 em todo o território nacional.\n\n- **Primeiro"
        " Turno:** Realizado no primeiro domingo de outubro de 2026 (04 de"
        " outubro de 2026).\n- **Segundo Turno:** Realizado no último domingo de"
        " outubro de 2026 (25 de outubro de 2026), apenas para os cargos de"
        " Presidente e Governador em unidades federativas onde nenhum candidato"
        " atingir a maioria absoluta dos votos válidos.\n- **Fechamento do"
        " Cadastro Eleitoral:** 151 dias antes do pleito. Nenhum requerimento de"
        " alistamento, transferência ou regularização do título de eleitor"
        " será aceito após esta data.\n- **Registro de Candidaturas:** O prazo"
        " limite para a transmissão dos dados dos candidatos pelos partidos e"
        " federações ao TSE encerra-se às 19h do dia 15 de agosto de 2026.\n-"
        " **Diplomação:** Os candidatos eleitos devem ser diplomados pela"
        " Justiça Eleitoral até o dia 18 de dezembro de 2026.\n"
    ),
    "cargos_e_votação.md": (
        "# Eleições Gerais 2026 · Guia Eleitoral · Seção 2 · Cargos e Ordem de"
        " Votação\n\n## 2.1 Ordem de votação na urna\n\nNas Eleições de 2026, a"
        " ordem de votação no teclado da Urna Eletrônica segue rigorosamente a"
        " sequência determinada pelo TSE:\n\n1. **Deputado Federal** (4"
        " dígitos)\n2. **Deputado Estadual ou Distrital** (5 dígitos)\n3."
        " **Senador — 1ª Vaga** (3 dígitos)\n4. **Senador — 2ª Vaga** (3"
        " dígitos)\n5. **Governador** (2 dígitos)\n6. **Presidente da"
        " República** (2 dígitos)\n\n## 2.2 Renovação do Senado\n\nEm 2026,"
        " haverá a renovação de **dois terços (2/3)** das cadeiras do Senado"
        " Federal. Cada eleitor deverá votar em dois candidatos diferentes para"
        " o Senado. Se o eleitor digitar o mesmo número de candidato para as"
        " duas vagas, o segundo voto será considerado nulo pela urna.\n"
    ),
    "urna_eletronica_specs.md": (
        "# Eleições Gerais 2026 · Manutenção Técnica · Seção 3 · Especificações"
        " da Urna (Modelo UE2026)\n\nO modelo UE2026 traz atualizações na"
        " arquitetura de hardware e segurança criptográfica para o pleito de"
        " 2026.\n\n- **Processador:** Módulo de alta velocidade com suporte a"
        " HSM (Hardware Security Module) integrado.\n- **Alimentação:** Bateria"
        " interna de Lítio-Ferro-Fosfato (LiFePO4) com autonomia mínima de 12"
        " horas de operação contínua sem energia externa.\n- **Mídia de"
        " Carga:** Cartão de memória interno encriptado com chave assimétrica"
        " RSA-4096 e algoritmo SHA-512.\n- **Impressora do BU (Boletim de"
        " Urna):** Módulo térmico acoplado que imprime 5 vias obrigatórias do"
        " Boletim de Urna ao encerramento da votação.\n- **Acessibilidade:**"
        " Teclado em Braille, sintetizador de voz aprimorado para deficientes"
        " visuais e tela auxiliar com intérprete virtual de Libras.\n- **Rede"
        " e Conectividade:** A urna opera sem nenhuma conexão física ou de"
        " rede (offline) durante todo o período de votação. A transmissão de"
        " dados é efetuada apenas após o término do pleito via rede privativa"
        " da Justiça Eleitoral.\n"
    ),
    "procedimentos_secao.md": (
        "# Eleições Gerais 2026 · Manual do Mesário · Seção 4 · Procedimentos de"
        " Seção e Votação\n\nEste documento define o fluxo operacional das"
        " seções eleitorais durante o dia de votação. Cada procedimento possui"
        " um código no registro operacional (Seção 9).\n\n## 4.1 Abertura da"
        " Seção e Emissão da Zerésima (Código PROC-AB1)\n\nÀs 07:00 (horário de"
        " Brasília), os mesários e o presidente de mesa realizam a conferência"
        " dos lacres da urna UE2026. Em seguida, a urna é ligada e emitida a"
        " **Zerésima**, documento que comprova que não há nenhum voto"
        " registrado na máquina para nenhum candidato. O presidente da mesa, os"
        " mesários e os fiscais de partidos presentes assinam o relatório da"
        " Zerésima.\n\n## 4.2 Habilitação do Eleitor e Biometria (Código"
        " PROC-BIO)\n\nO eleitor apresenta documento oficial com foto. O"
        " mesário insere o CPF ou número do título no terminal do mesário. O"
        " eleitor posiciona o polegar ou indicador no leitor biométrico. A urna"
        " permite até **4 tentativas** de leitura biométrica. Se todas"
        " falharem, o mesário pode liberar a votação mediante confirmação do"
        " ano de nascimento do eleitor e digitação de senha de autorização do"
        " presidente de mesa, registrando a ocorrência na ata da seção.\n\n##"
        " 4.3 Encerramento e Emissão de Boletins de Urna (Código PROC-ENC)\n\nÀs"
        " 17:00 (horário de Brasília), o presidente de mesa declara encerrada a"
        " votação, desde que não haja eleitores na fila (caso haja, são"
        " distribuídas senhas). É inserida a Mídia de Memória de Resultado"
        " (MRR) para finalizar o pleito e são impressas as **5 vias"
        " obrigatórias** do Boletim de Urna (BU). Uma via é afixada na porta da"
        " seção para consulta pública imediata e as demais seguem para a junta"
        " eleitoral junto com a mídia de gravação.\n\n## 4.4 Contingência e"
        " Troca de Urna (Código PROC-CONT)\n\nEm caso de falha irrecuperável do"
        " hardware da urna durante a votação, o suporte técnico é acionado. Se"
        " a reinicialização falhar, a urna é substituída por uma **Urna de"
        " Contingência**. O cartão de memória da urna com defeito é"
        " transferido para a nova urna, garantindo que nenhum voto gravado seja"
        " perdido.\n"
    ),
    "financiamento_e_prestacao.md": (
        "# Eleições Gerais 2026 · Manual de Prestação de Contas · Seção 5 ·"
        " Regras de Financiamento\n\n## 5.1 Fundo Especial de Financiamento de"
        " Campanha (FEFC)\n\nO Fundo Eleitoral é distribuído entre os partidos"
        " com base na representação parlamentar na Câmara dos Deputados e no"
        " Senado Federal. Os recursos devem ser mantidos em conta bancária"
        " específica e auditável, aberta exclusivamente para a campanha de"
        " 2026.\n\n## 5.2 Doações de Pessoas Físicas\n\nPessoas físicas podem"
        " doar para candidatos até o limite de **10% do rendimento bruto**"
        " declarado à Receita Federal no ano-calendário anterior (2025). Doações"
        " acima deste limite sujeitam o doador à multa de até 100% do valor"
        " excedente. Doações efetuadas via Pix só serão aceitas se a chave for"
        " o CPF do doador.\n\n## 5.3 Vedações Absolutas\n\nÉ estritamente"
        " proibido o financiamento de campanha por:\n- Pessoas jurídicas"
        " (empresas e corporações).\n- Entidades ou governos estrangeiros.\n-"
        " Concessionárias de serviços públicos e entidades sindicais.\n"
    ),
    "propaganda_e_midia.md": (
        "# Eleições Gerais 2026 · Legislação Eleitoral · Seção 6 · Propaganda e"
        " Redes Sociais\n\n## 6.1 Horário Gratuito de Rádio e TV\n\nO horário"
        " eleitoral gratuito começa 35 dias antes do antevéspera do primeiro"
        " turno. A divisão do tempo destina 10% de forma igualitária entre todos"
        " os partidos e 90% proporcionalmente ao número de representantes na"
        " Câmara dos Deputados.\n\n## 6.2 Regras para Inteligência Artificial e"
        " Deepfakes\n\nO uso de Inteligência Artificial em peças de propaganda"
        " exige **rotulagem explícita e clara** informando que o conteúdo foi"
        " gerado ou alterado sinteticamente. É terminantemente proibido o uso"
        " de *deepfakes* para simular a fala ou imagem de candidatos criando"
        " declarações falsas. A violação acarreta o cancelamento do registro de"
        " candidatura e inelegibilidade do responsável.\n\n## 6.3"
        " Impulsionamento de Conteúdo\n\nO impulsionamento de publicações na"
        " internet só é permitido se realizado diretamente nas plataformas de"
        " redes sociais e contratado exclusivamente por candidatos, partidos ou"
        " coligações, sendo obrigatório a identificação com a marca \"Propaganda"
        " Eleitoral\". É proibido o uso de disparos em massa via aplicativos de"
        " mensagaria sem consentimento prévio do destinatário.\n"
    ),
    "codigos_procedimento.md": (
        "# Eleições Gerais 2026 · Guia Operacional · Seção 9 · Tabela de"
        " Códigos de Procedimento\n\nCódigos de referência padronizados para"
        " comunicação oficial entre seções eleitorais, cartórios e juntas do"
        " TRE.\n\n| Código | Procedimento | Responsável pela Execução |\n|---|"
        "---|---|\n| PROC-AB1 | Emissão de Zerésima e Abertura | Presidente da"
        " Mesa |\n| PROC-BIO | Habilitação Biométrica do Eleitor | 1º Mesário"
        " |\n| PROC-ENC | Encerramento e Impressão do BU | Presidente da Mesa"
        " |\n| PROC-CONT | Substituição por Urna de Contingência | Técnico do"
        " TRE |\n| PROC-APUR | Transmissão de Dados do Resultado | Operador de"
        " Transmissão |\n| PROC-AUD | Auditoria de Urnas por Amostragem | Comissão"
        " de Auditoria |\n\nQualquer atualização nas rotinas operacionais gera"
        " a revisão do sufixo do código (exemplo: PROC-AB1 atualizado passa a"
        " ser PROC-AB2).\n"
    ),
    "glossario_eleitoral.md": (
        "# Eleições Gerais 2026 · Manual de Comunicação · Seção 7 · Glossário de"
        " Termos Oficiais\n\nTermos padronizados utilizados pela Justiça"
        " Eleitoral durante a transmissão e apuração de dados:\n\n-"
        " **\"Zerésima.\"** Relatório impresso pela Urna Eletrônica antes do"
        " início da votação demonstrando que o total de votos acumulados é zero.\n-"
        " **\"Boletim de Urna (BU).\"** Documento oficial impresso ao final da"
        " votação que registra o total de votos recebidos por candidato,"
        " brancos e nulos em uma seção.\n- **\"Quociente Eleitoral.\"** Resultado"
        " da divisão do número de votos válidos pelo número de vagas a preencher"
        " na eleição proporcional (Deputados).\n- **\"Mídia de Memória de\""
        " Resultado (MRR).\"** Dispositivo de armazenamento seguro usado para"
        " transportar os dados da urna para o ponto de transmissão.\n-"
        " **\"Voto de Legenda.\"** Voto digitado apenas com o número do partido"
        " na eleição proporcional, atribuído à sigla e não a um candidato"
        " específico.\n"
    ),
    "fiscalizacao_e_auditoria.md": (
        "# Eleições Gerais 2026 · Segurança da Votação · Seção 8 · Auditoria e"
        " Fiscalização\n\n## 8.1 Teste de Integridade (Votação Paralela)\n\nNo"
        " mesmo dia da eleição, o TRE sorteia seções eleitorais em todo o"
        " Estado para a realização do **Teste de Integridade em Tempo Real**."
        " As urnas sorteadas são trazidas à sede do TRE e submetidas a uma"
        " votação simulada simultânea à votação oficial, com digitação de"
        " cédulas físicas preenchidas por representantes de entidades da"
        " sociedade civil, sendo tudo gravado em áudio e vídeo.\n\n## 8.2"
        " Auditoria do Código-Fonte\n\nOs sistemas e códigos-fonte da Urna"
        " Eletrônica são abertos para inspeção por partidos políticos, Forças"
        " Armadas, OAB, Polícia Federal e universidades até 12 meses antes do"
        " pleito na sala de apuração do Tribunal Superior Eleitoral.\n"
    ),
}

# Gravando os arquivos Markdown
for name, text in CORPUS.items():
    file_path = os.path.join(CORPUS_DIR, name)
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(text)

print(f"{len(CORPUS)} documentos gravados no diretório '{CORPUS_DIR}'")

9 documentos gravados no diretório 'corpus/md'


#Chunking

In [8]:
!pip install -q langchain-text-splitters

In [9]:
import os
from langchain_text_splitters import (
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
)

def chunking(folder=CORPUS_DIR, chunk_size=512, overlap=50):
    headers_to_split_on = [
        ("#", "title"),
        ("##", "subtitle"),
        ("###", "section"),
    ]

    markdown_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=headers_to_split_on, strip_headers=False
    )

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=overlap
    )

    rows = []

    for filename in sorted(os.listdir(folder)):
        if not filename.endswith(".md"):
            continue

        filepath = os.path.join(folder, filename)
        with open(filepath, encoding="utf-8") as f:
            text = f.read()

        docs_md = markdown_splitter.split_text(text)

        chunks = text_splitter.split_documents(docs_md)
        for i, c in enumerate(chunks):
            section_name = (
                c.metadata.get("section")
                or c.metadata.get("subtitle")
                or c.metadata.get("title")
                or "general"
            )

            rows.append({
                "id": f"{filename}#{i}",
                "text": c.page_content,
                "doc": filename,
                "chunk": i,
                "section": section_name,
            })

    return rows

In [10]:
def build_index(rows, name, persist_dir=CHROMA_DIR, batch=32, rebuild=False):
    store = chromadb.PersistentClient(path=persist_dir)

    if rebuild:
        try:
            store.delete_collection(name)
        except Exception:
            pass

    col = store.get_or_create_collection(
        name,
        metadata={"hnsw:space": "cosine", "embedding_model": EMB_MODEL},
    )

    if col.count() == len(rows):
        return col

    for i in range(0, len(rows), batch):
        part = rows[i : i + batch]
        col.upsert(
            ids=[r["id"] for r in part],
            documents=[r["text"] for r in part],
            embeddings=embed([r["text"] for r in part]),
            metadatas=[
                {"doc": r["doc"], "chunk": r["chunk"], "section": r["section"]}
                for r in part
            ],
        )
    return col

In [11]:
def sources_used(hits):
    """A lista de fontes vem da recuperação e monta o formato [doc · section]."""
    seen, out = set(), []
    for h in hits:
        key = f"{h['doc']} · {h['section']}"
        if key not in seen:
            seen.add(key)
            out.append(key)
    return out

In [12]:
# Processando os documentos em chunks e construindo o índice no ChromaDB
rows = chunking()
print(f"Total de chunks gerados: {len(rows)}")

collection_name = "eleicoes_2026_col"
col = build_index(rows, collection_name, rebuild=True)
print(f"Índice vetorial construído com sucesso! Total de itens na coleção: {col.count()}")

Total de chunks gerados: 26
Índice vetorial construído com sucesso! Total de itens na coleção: 26


In [13]:
def ask_rag(question, k=4):
    # 1. Recuperar os chunks mais relevantes do ChromaDB
    hits = retrieve(col, question, k=k)

    # 2. Montar o contexto com os textos recuperados e suas respectivas fontes
    context_parts = []
    for h in hits:
        context_parts.append(f"Documento: {h['doc']} (Seção: {h['section']})\nConteúdo: {h['text']}")
    context = "\n\n---\n\n".join(context_parts)

    # 3. Construir o prompt do sistema e do usuário para o LLM
    system_prompt = (
        "Você é um assistente especialista nas Eleições Gerais de 2026. "
        "Responda à pergunta do usuário utilizando estritamente o contexto fornecido abaixo. "
        "Se a resposta não puder ser encontrada no contexto, informe que não possui informações suficientes."
    )

    user_prompt = f"""Contexto:\n{context}\n\nPergunta: {question}\nResposta:"""

    # 4. Chamar o modelo Qwen 2.5 via API do Ollama (cliente OpenAI)
    response = client.chat.completions.create(
        model=GEN_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.1
    )

    answer = response.choices[0].message.content
    sources = sources_used(hits)

    return answer, sources

In [16]:
# Realize sua consulta aqui
pergunta = "Qual a data do segundo turno?"

resposta, fontes = ask_rag(pergunta, k=3)

print(f"📌 Pergunta: {pergunta}\n")
print(f"🤖 Resposta:\n{resposta}\n")
print("📚 Fontes Consultadas:")
for fonte in fontes:
    print(f"  - {fonte}")

📌 Pergunta: Qual a data do segundo turno?

🤖 Resposta:
O segundo turno das Eleições Gerais de 2026 será realizado no último domingo de outubro de 2026, especificamente no dia 25 de outubro de 2026.

📚 Fontes Consultadas:
  - datas_e_prazos.md · Eleições Gerais 2026 · Guia de Calendário · Seção 1 · Datas e Prazos Oficiais
  - procedimentos_secao.md · Eleições Gerais 2026 · Manual do Mesário · Seção 4 · Procedimentos de Seção e Votação


#Questions

In [17]:
import json
import os

os.makedirs("results", exist_ok=True)

QUESTIONS = [
    {
        "id": "q1",
        "tier": "direct",
        "question": "Em que dia será realizado o primeiro turno das Eleições Gerais de 2026?",
        "must_contain": ["04 de outubro", "2026"],
        "gold_docs": ["datas_e_prazos.md"],
        "expect_refusal": False
    },
    {
        "id": "q2",
        "tier": "direct",
        "question": "Qual é a ordem de votação no teclado da Urna Eletrônica em 2026?",
        "must_contain": ["Deputado Federal", "Presidente"],
        "gold_docs": ["cargos_e_votação.md"],
        "expect_refusal": False
    },
    {
        "id": "q3",
        "tier": "multi",
        "question": "Quais são os limites de doação de pessoas físicas e quais as restrições sobre doações via Pix?",
        "must_contain": ["10%", "rendimento bruto", "cpf"],
        "gold_docs": ["financiamento_e_prestacao.md"],
        "expect_refusal": False
    },
    {
        "id": "q4",
        "tier": "none",
        "question": "Qual é a previsão do tempo para o dia da eleição em São Paulo?",
        "must_contain": [],
        "gold_docs": [],
        "expect_refusal": True
    }
]

with open("questions.json", "w", encoding="utf-8") as f:
    json.dump(QUESTIONS, f, ensure_ascii=False, indent=2)

print("Arquivo questions.json criado com sucesso!")

Arquivo questions.json criado com sucesso!


#Técnicas

In [18]:
import math
from collections import defaultdict

def rrf(rankings, c=60):
    """Reciprocal Rank Fusion para combinar múltiplos rankings."""
    fused = defaultdict(float)
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking, 1):
            fused[doc_id] += 1.0 / (c + rank)
    return sorted(fused.items(), key=lambda x: -x[1])

# --- 1. Step-Back Retrieval ---
def retrieve_stepback(col, question, k=4):
    resp = client.chat.completions.create(
        model=GEN_MODEL,
        messages=[
            {"role": "system", "content": "You are an expert at query reformulation. Write a broader, more abstract step-back question that captures underlying principles."},
            {"role": "user", "content": f"Question: {question}"}
        ],
        temperature=0.0
    )
    stepback_q = resp.choices[0].message.content.strip()

    hits_orig = retrieve(col, question, k=k)
    hits_step = retrieve(col, stepback_q, k=k)

    seen = {}
    for h in hits_orig + hits_step:
        key = f"{h['doc']}#{h.get('section','general')}"
        if key not in seen or h['distance'] < seen[key]['distance']:
            seen[key] = h

    combined = sorted(seen.values(), key=lambda x: x['distance'])
    return combined[:k]

# --- 2. Multi-Query / RAG-Fusion ---
def retrieve_multiquery(col, question, k=4, n_queries=3):
    resp = client.chat.completions.create(
        model=GEN_MODEL,
        messages=[
            {"role": "system", "content": f"You are a search query generator. Generate {n_queries} alternative search queries variations for the given question, one per line. Output only the queries."},
            {"role": "user", "content": question}
        ],
        temperature=0.3
    )
    queries = [q.strip() for q in resp.choices[0].message.content.splitlines() if q.strip()]
    queries = [question] + queries[:n_queries]

    rankings_ids = []
    hit_map = {}

    for q in queries:
        res = col.query(query_embeddings=embed([q]), n_results=k, include=["documents", "metadatas", "distances", "ids"])
        q_ids = res["ids"][0]
        rankings_ids.append(q_ids)
        for i, t, m, d in zip(q_ids, res["documents"][0], res["metadatas"][0], res["distances"][0]):
            hit_map[i] = {"id": i, "text": t, "doc": m["doc"], "section": m["section"], "distance": d}

    fused_scores = rrf(rankings_ids)
    final_hits = []
    for doc_id, score in fused_scores[:k]:
        if doc_id in hit_map:
            h = hit_map[doc_id]
            h["rrf_score"] = score
            final_hits.append(h)

    return final_hits

# --- 3. RAG with Memory ---
class RAGMemory:
    def __init__(self):
        self.history = []

    def add_turn(self, role, content):
        self.history.append({"role": role, "content": content})

    def get_contextualized_query(self, current_question):
        if not self.history:
            return current_question
        messages = [
            {"role": "system", "content": "Given the conversation history and a follow-up question, rewrite the follow-up question to be standalone and fully contextualized."},
        ]
        for turn in self.history[-4:]:
            messages.append(turn)
        messages.append({"role": "user", "content": f"Follow-up question: {current_question}\nStandalone question:"})

        resp = client.chat.completions.create(model=GEN_MODEL, messages=messages, temperature=0.0)
        return resp.choices[0].message.content.strip()

def ask_rag_with_memory(col, question, memory, k=4):
    standalone_q = memory.get_contextualized_query(question)
    hits = retrieve(col, standalone_q, k=k)

    context_parts = [f"Documento: {h['doc']} (Seção: {h['section']})\nConteúdo: {h['text']}" for h in hits]
    context = "\n\n---\n\n".join(context_parts)

    system_prompt = "Você é um assistente especialista nas Eleições Gerais de 2026. Responda utilizando estritamente o contexto fornecido."

    messages = [{"role": "system", "content": system_prompt}]
    for turn in memory.history[-4:]:
        messages.append(turn)
    messages.append({"role": "user", "content": f"Contexto:\n{context}\n\nPergunta: {standalone_q}"})

    resp = client.chat.completions.create(model=GEN_MODEL, messages=messages, temperature=0.1)
    answer = resp.choices[0].message.content

    memory.add_turn("user", question)
    memory.add_turn("assistant", answer)

    return answer, sources_used(hits), standalone_q

In [21]:
import os
import time
import pandas as pd

os.makedirs("results", exist_ok=True)
global LLM_CALLS
LLM_CALLS = 0

def generate(messages, model=GEN_MODEL, temperature=0.0):
    global LLM_CALLS
    LLM_CALLS += 1
    resp = client.chat.completions.create(model=model, messages=messages, temperature=temperature)
    return resp.choices[0].message.content.strip()

REFUSAL_TERMS = ["não possui", "não encontrei", "não tenho", "não é possível", "the handbook does not cover this"]

def grade(answer, q):
    a = answer.lower()
    refused = any(term in a for term in REFUSAL_TERMS)
    if q["expect_refusal"]:
        return refused, refused
    matched = all(kw.lower() in a for kw in q["must_contain"])
    return matched and not refused, refused

# --- Implementação das Técnicas com contagem de LLM_CALLS ---
def run_variant_baseline(col, questions, k=4):
    records = []
    global LLM_CALLS
    for q in questions:
        LLM_CALLS = 0
        t0 = time.time()
        hits = retrieve(col, q["question"], k=k)
        context = "\n\n".join(f"[{h['doc']} · {h['section']}]\n{h['text']}" for h in hits)
        messages = [
            {"role": "system", "content": "Você é um assistente especialista nas Eleições de 2026. Responda usando APENAS o contexto abaixo. Se não souber, diga que não possui informações suficientes."},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {q['question']}"}
        ]
        answer = generate(messages)
        total = time.time() - t0
        correct, refused = grade(answer, q)

        records.append({
            "label": "baseline", "id": q["id"], "tier": q["tier"], "question": q["question"],
            "n_hits": len(hits), "answer": answer, "correct": correct, "refused": refused,
            "llm_calls": LLM_CALLS, "latency_s": round(total, 2)
        })
    df = pd.DataFrame(records)
    df.to_csv("results/results_baseline.csv", index=False)
    return df

def run_variant_stepback(col, questions, k=4):
    records = []
    global LLM_CALLS
    for q in questions:
        LLM_CALLS = 0
        t0 = time.time()

        # 1. Step-Back query generation (1 chamada LLM)
        stepback_msgs = [
            {"role": "system", "content": "Write a broader, more abstract step-back question that captures underlying principles."},
            {"role": "user", "content": f"Question: {q['question']}"}
        ]
        stepback_q = generate(stepback_msgs)

        # Recuperação combinada original + step-back
        hits_orig = retrieve(col, q["question"], k=k)
        hits_step = retrieve(col, stepback_q, k=k)
        seen = {f"{h['doc']}#{h['section']}": h for h in hits_orig + hits_step}
        hits = sorted(seen.values(), key=lambda x: x['distance'])[:k]

        context = "\n\n".join(f"[{h['doc']} · {h['section']}]\n{h['text']}" for h in hits)
        messages = [
            {"role": "system", "content": "Você é um assistente especialista nas Eleições de 2026. Responda usando APENAS o contexto abaixo."},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {q['question']}"}
        ]
        answer = generate(messages) # Geração final (2ª chamada LLM)
        total = time.time() - t0
        correct, refused = grade(answer, q)

        records.append({
            "label": "stepback", "id": q["id"], "tier": q["tier"], "question": q["question"],
            "n_hits": len(hits), "answer": answer, "correct": correct, "refused": refused,
            "llm_calls": LLM_CALLS, "latency_s": round(total, 2)
        })
    df = pd.DataFrame(records)
    df.to_csv("results/results_stepback.csv", index=False)
    return df

def run_variant_multiquery(col, questions, k=4):
    records = []
    global LLM_CALLS
    for q in questions:
        LLM_CALLS = 0
        t0 = time.time()

        # 1. Multi-query generation (1 chamada LLM)
        mq_msgs = [
            {"role": "system", "content": "Generate 2 alternative search query variations for the given question, one per line. Output only the queries."},
            {"role": "user", "content": q['question']}
        ]
        queries_text = generate(mq_msgs)
        queries = [q['question']] + [line.strip() for line in queries_text.splitlines() if line.strip()][:2]

        # RRF / Fusão simples
        hit_map = {}
        for query in queries:
            for h in retrieve(col, query, k=k):
                hit_map[h['doc'] + h['section']] = h
        hits = list(hit_map.values())[:k]

        context = "\n\n".join(f"[{h['doc']} · {h['section']}]\n{h['text']}" for h in hits)
        messages = [
            {"role": "system", "content": "Você é um assistente especialista nas Eleições de 2026. Responda usando APENAS o contexto abaixo."},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {q['question']}"}
        ]
        answer = generate(messages) # Geração final (2ª chamada LLM)
        total = time.time() - t0
        correct, refused = grade(answer, q)

        records.append({
            "label": "multiquery", "id": q["id"], "tier": q["tier"], "question": q["question"],
            "n_hits": len(hits), "answer": answer, "correct": correct, "refused": refused,
            "llm_calls": LLM_CALLS, "latency_s": round(total, 2)
        })
    df = pd.DataFrame(records)
    df.to_csv("results/results_multiquery.csv", index=False)
    return df

# Executando e gerando os arquivos na pasta results/
with open("questions.json", encoding="utf-8") as f:
    questions_data = json.load(f)

print("Executando Baseline...")
df_base = run_variant_baseline(col, questions_data)
print("Executando Step-Back...")
df_step = run_variant_stepback(col, questions_data)
print("Executando Multi-Query...")
df_multi = run_variant_multiquery(col, questions_data)

print("\nTodos os testes executados com sucesso e arquivos salvos em 'results/'!")

Executando Baseline...
Executando Step-Back...
Executando Multi-Query...

Todos os testes executados com sucesso e arquivos salvos em 'results/'!
